# Proof of Absence from Vehicle Radar: reproduction notebook (v2.3.1)

**Private until the patent decision and employer IP clearance. Do not share.**

The notebook does not modify any shipped file: everything it recomputes is written to a temporary folder, compared with the shipped files and deleted at the end.

| What | Status |
|---|---|
| Every rate, count, interval and p-value in the paper | **Simulated** (2-D ray-traced radar simulator, `sim.py`) |
| Sensor parameters (radar cross-section, fading, fault rates) | **Assumed** (Table II of the paper) |
| Geometry round trip, known answers, ledger attack detection, verdict equality after disclosure | **Shown in code** (on simulated records) |
| Real-radar performance | **Not shown**: no real sensor data are used |

Run time: about 3 minutes (Python >= 3.12 for the pinned versions). In Colab, restart the runtime once after the first cell has installed the pinned versions, then run all cells again.

In [ ]:
# 1. Setup: locate the package. In Colab: set REPO_URL to your GitHub copy (a private repository needs a
#    token: https://<token>@github.com/<user>/<repo>.git), or leave it empty and upload the package zip when asked.
REPO_URL = ''
import os, sys, subprocess, zipfile, tempfile, shutil, json, hashlib, filecmp, importlib.metadata as md
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'; sys.dont_write_bytecode = True     # leave no __pycache__ in the package
IN_COLAB = 'google.colab' in sys.modules
is_pkg = lambda c: os.path.exists(os.path.join(c, 'sim.py')) and os.path.exists(os.path.join(c, 'results73'))
PKG = next((os.path.abspath(c) for c in ['.', '..'] if is_pkg(c)), None)
if PKG is None and IN_COLAB:
    PKG = next((r for r, _, _ in os.walk('/content') if is_pkg(r)), None)
    if PKG is None and REPO_URL:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/PAL-proof-of-absence'], check=True)
        PKG = '/content/PAL-proof-of-absence'
    if PKG is None:
        from google.colab import files
        up = files.upload(); zipfile.ZipFile(next(iter(up))).extractall('/content/pal')
        PKG = next((r for r, _, _ in os.walk('/content/pal') if is_pkg(r)), None)
assert PKG, 'Run this notebook from the package folder, set REPO_URL, or upload the package zip.'
os.chdir(PKG); sys.path.insert(0, PKG)
PRE_EXISTING = {d for d in ('results7', '__pycache__') if os.path.exists(d)}
# pinned versions (requirements.txt): install if they differ; a restart is needed if numpy was already loaded
pins = dict(l.strip().split('==') for l in open('requirements.txt') if '==' in l and not l.startswith('#'))
have = {k: (md.version(k) if any(d.metadata['Name'].lower() == k for d in md.distributions()) else None) for k in pins}
if any(have[k] != v for k, v in pins.items()):
    print('installing pinned versions:', {k: f"{have[k]} -> {v}" for k, v in pins.items() if have[k] != v})
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    if 'numpy' in sys.modules:
        raise SystemExit('Pinned packages installed. Restart the runtime (Runtime > Restart session) and run all cells again.')
TMP = tempfile.mkdtemp(prefix='pal_check_')
import numpy, cryptography, matplotlib
print('python', sys.version.split()[0], '| numpy', numpy.__version__, '| cryptography', cryptography.__version__)

In [ ]:
# 2. Is this the frozen code? Compare SHA-256 with the pre-registration record
pre = json.load(open('results73/PREREGISTRATION.json'))
frozen = dict(pre['sha256']); frozen['sim.py'] = pre['amendments'][-1]['sim.py_sha256']   # disclosed amendment (G3 crash fix)
print('frozen at', pre['frozen_utc'])
for f, h in frozen.items():
    ok = hashlib.sha256(open(f, 'rb').read()).hexdigest() == h
    print(f'{f:15s}', 'MATCH' if ok else 'DIFFERENT'); assert ok, f + ' differs from the frozen code'

In [ ]:
# 3. Recompute every paper number from the result files, in a scratch copy, and compare
def scratch_run(script, outputs, args=()):
    work = os.path.join(TMP, os.path.basename(script) + '_run'); shutil.copytree(PKG, work, ignore=shutil.ignore_patterns('notebooks', 'paper', '__pycache__'))
    subprocess.run([sys.executable, script, *args], cwd=work, check=True, stdout=subprocess.DEVNULL)
    return {o: filecmp.cmp(os.path.join(work, o), os.path.join(PKG, o), shallow=False) for o in outputs}
checks = {'analyze_prereg.py': scratch_run('analyze_prereg.py', ['results73/prereg_analysis.json']),
          'make_numbers.py': scratch_run('make_numbers.py', ['numbers_rev2.json']),
          'round2/analyses.py': scratch_run('round2/analyses.py', ['round2/analyses.json']),
          'round2/analyses_r3.py': scratch_run('round2/analyses_r3.py', ['round2/analyses_r3.json'])}
for k, v in checks.items():
    print(f'{k:22s} identical:', v); assert all(v.values()), k
N = json.load(open('numbers_rev2.json')); m = N['main']['methods']
print(f"\nMain benchmark: {N['main']['genuine']} genuine, {N['main']['empty']} empty, {N['main']['neither']} neither")
for k, lab in [('Rule k3', 'rule tier'), ('PAL v2.3', 'formal tier'), ('Combined (formal and rule)', 'graded, standard'),
               ('ObjList-best v2.1', 'persistence-only baseline'), ('PAL-v1 (Phase 3)', 'early prototype')]:
    print(f"{lab:32s} WR {m[k]['WR']:3d}  WS {m[k]['WS']:3d}  coverage {100*m[k]['cov']:5.1f}%")

In [ ]:
# 4. The pre-registered test (H1, H2), the review statistics (paired CI, calibration), the matched-coverage comparison
# and the exploratory stress tests (round3/stress_x.json is shown as shipped; re-running it takes about 40 minutes)
P = json.load(open('results73/prereg_analysis.json')); A = json.load(open('round2/analyses.json')); A3 = json.load(open('round2/analyses_r3.json'))
print('H1:', P['H1']); print('H2:', P['H2'])
pr = A['prereg']; print('paired difference formal - rule: %.4f pp, 95%% CI [%.4f, %.4f] pp' % (100*pr['diff_formal_minus_rule'], 100*pr['diff_ci95'][0], 100*pr['diff_ci95'][1]))
print('claims with bound <= eps, observed vs expected misses:',
      {c: (v['bound_le_eps']['observed'], round(v['bound_le_eps']['expected'], 3)) for c, v in A3['calib_by_class'].items()})
print('cars, all claims, observed vs expected:', {c: (v['car']['observed'], round(v['car']['expected'], 3)) for c, v in A3['calib_by_class'].items()})
print('motorcycles, all claims:               ', {c: (v['motorcycle']['observed'], round(v['motorcycle']['expected'], 1)) for c, v in A3['calib_by_class'].items()})
print('matched coverage:', A3['matched_coverage']['rule_tier'], 'vs formal max', A3['matched_coverage']['formal_max_coverage'])
assert (P['H1']['formal_WR'], P['H1']['rule_WR'], P['H1']['p_exact_mcnemar']) == (5, 6, 1.0) and P['H2']['pass']
CC = A3['prereg_coverage_clustered']
print('pre-registered coverage, scene-clustered 95% CI:', {k: [round(100 * v['point'], 1)] + [round(100 * x, 1) for x in v['ci95']] for k, v in CC.items()})
assert [round(100 * x, 1) for x in CC['rule']['ci95'] + CC['formal']['ci95'] + CC['standard']['ci95']] == [69.7, 72.6, 42.5, 45.1, 40.1, 42.7]
assert sum(v['formal_refuted'] for v in A3['prereg_neither'].values()) == 9 and sum(v['rule_refuted'] for v in A3['prereg_neither'].values()) == 25
SX = json.load(open('round3/stress_x.json'))
assert [SX['occlusion'][m]['WR'] for m in ('formal', 'rule', 'persistence_only')] == [0, 1, 33]
assert round(100 * SX['occlusion']['rule']['coverage'], 1) == 85.6 and SX['occlusion']['formal']['coverage'] == 0.0
assert SX['brake-far-edge']['persistence_only']['WR'] == 204 and SX['atten10']['formal']['WR'] == SX['atten10']['rule']['WR'] == 0
T = json.load(open('round2/timing.json')); print('verifier time per claim (shipped, wall clock):', {k: round(v, 3) for k, v in T['verdict_time_s'].items()})
for k in ('occlusion', 'brake-far-edge', 'atten10'):
    print(f"stress {k:15s} WR formal/rule/persistence-only:", [SX[k][m]['WR'] for m in ('formal', 'rule', 'persistence_only')],
          ' coverage:', [round(100 * SX[k][m]['coverage'], 1) for m in ('formal', 'rule', 'persistence_only')])

In [ ]:
# 5. Regenerate scenes from their seeds and compare with the stored verdicts (bit-identical expected)
import csv, run_v23, run_v23_bench
def check(tag, job, n, cols):
    rows = {int(r['scene']): r for r in csv.DictReader(open(f'results73/{tag}.csv'))}
    bad = sum(str(new[c]) != rows[i][c] for i in range(n) for new in [job(i)] for c in cols)
    print(f'{tag:18s} {n} scenes x {len(cols)} verdict columns: {bad} differences'); assert bad == 0
check('main', lambda i: run_v23.job((i, 7_000_000, {}, True)), 60, ['PAL v2.3', 'Rule k3', 'ObjList-best v2.1', 'v2.3 no bound', 'PAL-v1 (Phase 3)'])
check('G1-rcs-12', lambda i: run_v23.job((i, 11_000_000, dict(run_v23.G)['G1-rcs-12'], False)), 60, ['PAL v2.3', 'Rule k3'])
check('confirm-degraded', lambda i: run_v23_bench.light((i, 10_000_000, dict(run_v23.CONFIRM)['confirm-degraded'])), 60, ['PAL v2.3', 'Rule k3', 'ObjList-best v2.1'])
if 'results7' not in PRE_EXISTING and os.path.isdir('results7') and not os.listdir('results7'): os.rmdir('results7')   # created by importing run_phase7

In [ ]:
# 6. Integrity ledger: verdict equality after selective disclosure + eight attacks (3 scenes), run in a scratch copy
#    (the script itself asserts that every attack is detected), and the sanity checks V1-V5 of the simulator (about 1 min)
work = os.path.join(TMP, 'r20_run'); shutil.copytree(PKG, work, ignore=shutil.ignore_patterns('notebooks', 'paper', '__pycache__'))
out = subprocess.run([sys.executable, 'batch3/r20_integrity.py', '3'], cwd=work, capture_output=True, text=True, check=True).stdout
print(out); assert 'identical after disclosure+verification: 3/3' in out
subprocess.run([sys.executable, 'validation/run_validation_v21.py'], cwd=work, check=True, stdout=subprocess.DEVNULL)
V = json.load(open(os.path.join(work, 'validation', 'validation_v21.json')))
print('V1 known answers pass:', all(c['pass'] for c in V['V1_known_answers']),
      '| V2 ideal sensors: false refutations', V['V2_ideal_sensors']['false_refutations'], 'false supports', V['V2_ideal_sensors']['false_supports'])
assert all(c['pass'] for c in V['V1_known_answers']) and V['V2_ideal_sensors']['false_refutations'] == 0

In [ ]:
# 7. Redraw the data figures of the paper into the scratch folder and compare with the shipped PDFs
if os.path.exists('paper/figs.py'):
    env = dict(os.environ, PAL_FIGDIR=TMP, PAL_ROOT=PKG)
    subprocess.run([sys.executable, 'paper/figs.py'], env=env, check=True, stdout=subprocess.DEVNULL)
    for f in ['fig_pie.pdf', 'fig_degr.pdf', 'fig_errcov.pdf', 'fig_calib.pdf']:
        print(f, 'identical' if filecmp.cmp(os.path.join(TMP, f), os.path.join('paper', f), shallow=False)
              else 'DIFFERENT bytes (expected with other fonts or matplotlib versions; compare visually)')   # soft check
else:
    print('paper/figs.py not in this copy; skipped')

In [ ]:
# 8. Clean up: delete the scratch folder and anything this notebook created in the package
shutil.rmtree(TMP, ignore_errors=True)
for d in ('results7', '__pycache__'):
    if d not in PRE_EXISTING and os.path.isdir(d): shutil.rmtree(d)
print('scratch folder removed; package unchanged')

## Full re-run (optional; hours on 2 cores)

Re-runs **overwrite** the shipped result files without asking. Run them in a copy (`cp -r PAL-proof-of-absence rerun && cd rerun`) and compare with `python3 compare_results.py ../PAL-proof-of-absence .`.

| Command | Produces | Approx. time (2 cores) |
|---|---|---|
| `python3 run_v23.py prereg` | `results73/G1…G4.csv` (pre-registered test) | 45 min |
| `python3 run_v23.py boundary` | `results73/boundary-far-edge.csv` | 8 min |
| `python3 run_v23.py main`, `python3 run_v23.py sweeps` | main benchmark, 13 degradation conditions (these modes ignore N) | 16 min, 40 min |
| `python3 run_v23_bench.py confirm` | stress sets (exploratory; the shipped files; not `run_v23.py confirm`) | 25 min |
| `python3 batch3/r6_bound_vs_sim.py 4000` | bound calibration | 15 min |
| `python3 batch3/r9_attacks.py 3000` | owner attacks | 10 min |
| `python3 batch3/r20_integrity.py 60 > batch3/r20_result.txt` | ledger test (8 attacks) | 12 min |
| `python3 review11/boundary_v21.py` | earlier free test on the far-edge scenes | 3 min |
| `python3 round2/analyses.py`, `python3 round2/analyses_r3.py` | review statistics (`timing.json` varies) | 1 min, 5 s |
| `python3 round2/sens_eq6.py` | exploratory sensitivity of (6) with a tightened no-return clause | about 3 h |
| `python3 batch3/r19_curves.py` | curved-road diagnosis (ideal-sensor check) | 1 min |
| `python3 validation/run_validation_v21.py` | sanity checks V1-V5 (known answers, ideal sensors, geometry round trip) | 1 min |
| `python3 round3/stress_x.py 2000` | exploratory stress tests: occlusion, braking claimant, attenuation, correlated fades | 40 min |
| `python3 round3/log_cost.py` | storage estimate of the detection log | 1 min |
| `python3 analyze_prereg.py && python3 make_numbers.py` | `prereg_analysis.json`, `numbers_rev2.json` | 1 min |

Seeds: main 7,000,000 + i; degradation 8,000,000 + i; stress 10,000,000 + i; pre-registered 11,000,000 + i; far edge 12,000,000 + i; attacks 13,000,000 + i; calibration 14,000,000 + i.

*AI-use disclosure:* code, analysis and this notebook were developed with the assistance of an AI system (Claude, Anthropic); the author is responsible for the content.